# 04 — Experiment B2: Fine-tuned transfer to Ar-PlaqSegm1

This notebook evaluates whether the advantage of carotid-domain **EMA-based latent self-distillation inspired by I-JEPA** persists when the encoder is fine-tuned for downstream plaque segmentation. The primary comparison is **ImageNet initialization versus EMA-distillation initialization**, using downstream seeds 42, 43 and 44.

Historical internal names containing `IJEPA_clean_2840` refer to the EMA-distillation condition and are retained for provenance.

## 1. Environment and paths

In [ ]:
!pip -q install timm pandas pillow opencv-python


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os, random, copy, time, hashlib
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import timm
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
class Config:
    ARPLAQUE_ROOT = "/content/drive/MyDrive/dataset Ar-PlaqSegm1"
    PROJECT_DIR = "/content/drive/MyDrive/IJEPA_Carotid_ViTS16"
    CLEAN_RUN = os.path.join(PROJECT_DIR, "clean_2840_20260930")
    CHECKPOINT_DIR = os.path.join(CLEAN_RUN, "checkpoints")
    RESULT_DIR = os.path.join(CLEAN_RUN, "results")
    STATE_DIR = os.path.join(CLEAN_RUN, "resume_state")
    REVIEW_DIR = os.path.join(CLEAN_RUN, "review")

    ARCH = "vit_small_patch16_224"
    IMG_SIZE = 224
    BATCH_SIZE = 16
    DOWNSTREAM_SEEDS = [42, 43, 44]
    MAX_EPOCHS = 50
    PATIENCE = 10
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    POS_WEIGHT = 10.0

for p in [Config.CHECKPOINT_DIR, Config.RESULT_DIR, Config.STATE_DIR, Config.REVIEW_DIR]:
    os.makedirs(p, exist_ok=True)

RESULT_CSV = os.path.join(Config.RESULT_DIR, "experiment_B_clean_frozen.csv")
SUMMARY_CSV = os.path.join(Config.RESULT_DIR, "experiment_B_clean_summary_mean_sd.csv")
print("Clean output namespace:", Config.CLEAN_RUN)



Clean output namespace: /content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930


In [ ]:
# Historical B1 namespace retained for compatibility
# Permanent checkpoints + negative-case evaluation

B_DEFINITIVE_RESULT_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_frozen_with_negative_analysis.csv"
)

B_DEFINITIVE_SUMMARY_CSV = os.path.join(
    Config.RESULT_DIR,
    "experiment_B_definitive_summary_mean_sd.csv"
)

# Clean project root is the parent of resume_state/
CLEAN_ROOT = os.path.dirname(Config.STATE_DIR.rstrip("/"))

B_BEST_DIR = os.path.join(
    CLEAN_ROOT,
    "checkpoints",
    "experiment_B_definitive"
)

os.makedirs(B_BEST_DIR, exist_ok=True)

print("Clean root:")
print(CLEAN_ROOT)

print("\nDefinitive results:")
print(B_DEFINITIVE_RESULT_CSV)

print("\nPermanent best checkpoints:")
print(B_BEST_DIR)

Clean root:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930

Definitive results:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/results/experiment_B_definitive_frozen_with_negative_analysis.csv

Permanent best checkpoints:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/checkpoints/experiment_B_definitive


In [ ]:
# Experiment B2 — fine-tuning output namespace

import os

FT_ROOT = os.path.join(
    CLEAN_ROOT,
    "experiment_B_finetuning"
)

FT_RESULT_DIR = os.path.join(
    FT_ROOT,
    "results"
)

FT_CHECKPOINT_DIR = os.path.join(
    FT_ROOT,
    "checkpoints"
)

FT_STATE_DIR = os.path.join(
    FT_ROOT,
    "resume_state"
)

FT_HISTORY_DIR = os.path.join(
    FT_ROOT,
    "history"
)

for d in [
    FT_RESULT_DIR,
    FT_CHECKPOINT_DIR,
    FT_STATE_DIR,
    FT_HISTORY_DIR
]:
    os.makedirs(d, exist_ok=True)


FT_RESULT_CSV = os.path.join(
    FT_RESULT_DIR,
    "experiment_B_finetuning_results.csv"
)

FT_SUMMARY_CSV = os.path.join(
    FT_RESULT_DIR,
    "experiment_B_finetuning_summary_mean_sd.csv"
)

FT_AUDIT_CSV = os.path.join(
    FT_RESULT_DIR,
    "experiment_B_finetuning_checkpoint_audit.csv"
)


print("Experiment B2 fine-tuning namespace ready.")
print()
print("Root:")
print(FT_ROOT)

print("\nResults:")
print(FT_RESULT_CSV)

print("\nPermanent checkpoints:")
print(FT_CHECKPOINT_DIR)

print("\nResume states:")
print(FT_STATE_DIR)

print("\nHistory:")
print(FT_HISTORY_DIR)

Experiment B2 fine-tuning namespace ready.

Root:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning

Results:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/results/experiment_B_finetuning_results.csv

Permanent checkpoints:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/checkpoints

Resume states:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/resume_state

History:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/history


## 2. Dataset split and checkpoints

In [ ]:
IJEPA_CKPT = os.path.join(Config.CHECKPOINT_DIR, "ijepa_corrected_2840_encoder.pth")
MAE_CKPT   = os.path.join(Config.CHECKPOINT_DIR, "mae_corrected_2840_encoder.pth")
assert os.path.exists(IJEPA_CKPT), IJEPA_CKPT
assert os.path.exists(MAE_CKPT), MAE_CKPT

def audit_ssl_ckpt(path, label):
    ck = torch.load(path, map_location="cpu")

    # Architecture
    assert ck.get("arch") == "vit_small_patch16_224", \
        f"{label}: unexpected architecture: {ck.get('arch')}"

    # Corrected SSL pool size
    assert int(ck.get("n_ssl_images", -1)) == 2840, \
        f"{label}: wrong SSL image count"

    # Collect ALL available provenance metadata
    provenance = " | ".join([
        str(ck.get("source", "")),
        str(ck.get("ssl_sources", "")),
        str(ck.get("ssl_pool", "")),
        str(ck.get("manifest", ""))
    ]).lower()

    # Verify corrected B-mode pool
    assert "1740" in provenance and "cubs" in provenance, \
        f"{label}: CUBS source not verified: {provenance}"

    assert "1100" in provenance and "orvile" in provenance, \
        f"{label}: Orvile source not verified: {provenance}"

    # If explicit mask metadata exists, masks must be excluded
    if "orvile_masks_included" in ck:
        assert ck["orvile_masks_included"] is False, \
            f"{label}: Orvile masks were included"

    print(
        f"{label}: CLEAN checkpoint verified | "
        f"SSL images={ck['n_ssl_images']}"
    )

    return ck


_ij = audit_ssl_ckpt(IJEPA_CKPT, "I-JEPA")
_ma = audit_ssl_ckpt(MAE_CKPT, "MAE")

del _ij, _ma

VALID_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def build_pairs(split):
    d = os.path.join(dataset_root, split)
    files = [f for f in os.listdir(d) if f.lower().endswith(VALID_EXTS)]
    images = sorted(f for f in files if "_labeled" not in f.lower())
    pairs = []
    for img in images:
        stem = os.path.splitext(img)[0].strip()
        mask = f"{stem}_labeled.png"
        ip, mp = os.path.join(d,img), os.path.join(d,mask)
        if os.path.exists(mp):
            pairs.append((ip,mp))
    return pairs

dataset_root = Config.ARPLAQUE_ROOT

print("Ar-PlaqSegm1 root:")
print(dataset_root)

assert os.path.isdir(dataset_root), \
    f"Ar-PlaqSegm1 root not found: {dataset_root}"

# Locate the actual Ar-PlaqSegm1 dataset directory
base_root = Config.ARPLAQUE_ROOT

candidates = [
    os.path.join(base_root, d)
    for d in os.listdir(base_root)
    if os.path.isdir(os.path.join(base_root, d))
    and not d.startswith("__")
]

dataset_root = None

for d in candidates:
    if (
        os.path.isdir(os.path.join(d, "train"))
        and os.path.isdir(os.path.join(d, "test"))
    ):
        dataset_root = d
        break

assert dataset_root is not None, \
    f"Could not locate Ar-PlaqSegm1 train/test folders under {base_root}"

print("Ar-PlaqSegm1 dataset root:")
print(dataset_root)

print("\nFolders:")
print(os.listdir(dataset_root))

train_pairs_all = build_pairs("train")
test_pairs = build_pairs("test")

# Fixed deterministic 80/20 split, retained from the prior Experiment B protocol.
rng = random.Random(42)
rng.shuffle(train_pairs_all)
n_val = int(round(len(train_pairs_all)*0.20))
val_pairs = train_pairs_all[:n_val]
tr_pairs = train_pairs_all[n_val:]

def positive_count(pairs):
    return sum(np.asarray(Image.open(m).convert("L")).max() > 0 for _,m in pairs)

print("Train/Val/Test:", len(tr_pairs), len(val_pairs), len(test_pairs))
print("Positive-mask test images:", positive_count(test_pairs))

assert (len(tr_pairs),len(val_pairs),len(test_pairs)) == (394,99,48)
assert positive_count(test_pairs) == 24
print("\nPRE-FLIGHT DATA AUDIT PASSED")

# Permanent best-checkpoint saver for Experiment B

def save_B_best_checkpoint(model, kind, seed, best_epoch, best_val):
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    path = os.path.join(
        B_BEST_DIR,
        f"B_{safe_kind}_seed{seed}_frozen_best.pth"
    )

    torch.save(
        {
            "encoder": kind,
            "seed": seed,
            "best_epoch": int(best_epoch),
            "best_val_positive_dice": float(best_val),
            "arch": Config.ARCH,
            "model_state_dict": {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        },
        path
    )

    return path

print("Permanent checkpoint saver defined.")


I-JEPA: CLEAN checkpoint verified | SSL images=2840
MAE: CLEAN checkpoint verified | SSL images=2840
Ar-PlaqSegm1 root:
/content/drive/MyDrive/dataset Ar-PlaqSegm1
Ar-PlaqSegm1 dataset root:
/content/drive/MyDrive/dataset Ar-PlaqSegm1/Ar-PlaqSegm1 First Argentine database of B-mode ul

Folders:
['test', 'train', 'arplaqsegm1_original_file_manifest.csv']
Train/Val/Test: 394 99 48
Positive-mask test images: 24

PRE-FLIGHT DATA AUDIT PASSED
Permanent checkpoint saver defined.


In [ ]:
# Experiment B2 — fine-tuning configuration

FT_ENCODERS = [
    "ImageNet",
    "IJEPA_clean_2840"
]

FT_SEEDS = [42, 43, 44]

# Differential learning rates
FT_ENCODER_LR = 1e-5
FT_DECODER_LR = 1e-3

# Preserve frozen Experiment B training/evaluation settings
FT_WEIGHT_DECAY = 1e-4
FT_MAX_EPOCHS = 50
FT_PATIENCE = 10
FT_POS_WEIGHT = 10.0
FT_THRESHOLD = 0.5

expected_ft_runs = [
    (encoder, seed)
    for seed in FT_SEEDS
    for encoder in FT_ENCODERS
]

print("Experiment B2 configuration")
print("---------------------------")
print("Encoders:", FT_ENCODERS)
print("Seeds:", FT_SEEDS)
print("Expected runs:", len(expected_ft_runs))
print()
print("Encoder LR:", FT_ENCODER_LR)
print("Decoder LR:", FT_DECODER_LR)
print("Weight decay:", FT_WEIGHT_DECAY)
print("Max epochs:", FT_MAX_EPOCHS)
print("Patience:", FT_PATIENCE)
print("Positive weight:", FT_POS_WEIGHT)
print("Threshold:", FT_THRESHOLD)

Experiment B2 configuration
---------------------------
Encoders: ['ImageNet', 'IJEPA_clean_2840']
Seeds: [42, 43, 44]
Expected runs: 6

Encoder LR: 1e-05
Decoder LR: 0.001
Weight decay: 0.0001
Max epochs: 50
Patience: 10
Positive weight: 10.0
Threshold: 0.5


In [ ]:
# Negative-case false-positive evaluation

@torch.no_grad()
def negative_case_metrics(model, loader, threshold=0.5):

    model.eval()

    fp_image_flags = []
    fp_pixel_fractions = []
    n_negative = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        probs = torch.sigmoid(logits)
        pred = (probs >= threshold).float()

        for i in range(x.size(0)):

            # Ground-truth mask contains no plaque pixels
            if y[i].sum().item() == 0:

                n_negative += 1

                fp_pixels = pred[i].sum().item()
                total_pixels = pred[i].numel()

                fp_image_flags.append(
                    float(fp_pixels > 0)
                )

                fp_pixel_fractions.append(
                    fp_pixels / total_pixels
                )

    assert n_negative == 24, (
        f"Expected 24 negative test cases, "
        f"found {n_negative}"
    )

    fp_image_flags = np.asarray(fp_image_flags, dtype=float)
    fp_pixel_fractions = np.asarray(
        fp_pixel_fractions,
        dtype=float
    )

    return {
        "n_negative_test": n_negative,
        "negative_cases_with_any_fp":
            int(fp_image_flags.sum()),
        "false_positive_image_rate":
            float(fp_image_flags.mean()),
        "mean_false_positive_pixel_fraction":
            float(fp_pixel_fractions.mean()),
        "sd_false_positive_pixel_fraction":
            float(fp_pixel_fractions.std(ddof=1)),
        "median_false_positive_pixel_fraction":
            float(np.median(fp_pixel_fractions))
    }

print("Negative-case evaluator defined.")

Negative-case evaluator defined.


## 3. Paired image/mask preprocessing

Training uses a paired horizontal flip sampled from the downstream seed. The same geometric transform is applied to image and mask. Validation and test preprocessing are deterministic.

In [ ]:
IMAGENET_MEAN=(0.485,0.456,0.406)
IMAGENET_STD=(0.229,0.224,0.225)

class SegDataset(Dataset):
    def __init__(self, pairs, train=False):
        self.pairs=list(pairs)
        self.train=train
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,i):
        ip,mp=self.pairs[i]
        im=Image.open(ip).convert("RGB")
        ma=Image.open(mp).convert("L")

        # Paired geometric augmentation.
        if self.train and random.random() < 0.5:
            im=TF.hflip(im)
            ma=TF.hflip(ma)

        im=TF.resize(im,[Config.IMG_SIZE,Config.IMG_SIZE], interpolation=InterpolationMode.BILINEAR)
        ma=TF.resize(ma,[Config.IMG_SIZE,Config.IMG_SIZE], interpolation=InterpolationMode.NEAREST)

        im=TF.normalize(TF.to_tensor(im),IMAGENET_MEAN,IMAGENET_STD)
        ma=(TF.to_tensor(ma)>.5).float()
        return im,ma

def make_loaders(seed):
    # Recreate loaders for each seed so shuffling/augmentation are seed-controlled.
    set_seed(seed)
    g=torch.Generator()
    g.manual_seed(seed)
    train_loader=DataLoader(
        SegDataset(tr_pairs,train=True), batch_size=Config.BATCH_SIZE,
        shuffle=True, generator=g, num_workers=0, pin_memory=True
    )
    val_loader=DataLoader(
        SegDataset(val_pairs,train=False), batch_size=Config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    test_loader=DataLoader(
        SegDataset(test_pairs,train=False), batch_size=Config.BATCH_SIZE,
        shuffle=False, num_workers=0, pin_memory=True
    )
    return train_loader,val_loader,test_loader

print("Paired preprocessing ready.")


Paired preprocessing ready.


## 4. Fine-tuned encoder conditions and segmentation head

In [ ]:
ENCODERS = ["Random", "ImageNet", "MAE_clean_2840", "IJEPA_clean_2840"]

def build_encoder(kind, seed):
    set_seed(seed)
    if kind=="Random":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
    elif kind=="ImageNet":
        enc=timm.create_model(Config.ARCH, pretrained=True, num_classes=0)
    elif kind=="MAE_clean_2840":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        ck=torch.load(MAE_CKPT,map_location="cpu")
        enc.load_state_dict(ck["encoder_state_dict"],strict=True)
    elif kind=="IJEPA_clean_2840":
        enc=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        ck=torch.load(IJEPA_CKPT,map_location="cpu")
        enc.load_state_dict(ck["encoder_state_dict"],strict=True)
    else:
        raise ValueError(kind)

    for p in enc.parameters():
        p.requires_grad=False
    return enc.to(DEVICE).eval()

# EXPERIMENT B2 — TRAINABLE ENCODER

def build_finetune_encoder(kind, seed):
    set_seed(seed)

    if kind == "ImageNet":
        enc = timm.create_model(
            Config.ARCH,
            pretrained=True,
            num_classes=0
        )

    elif kind == "IJEPA_clean_2840":
        enc = timm.create_model(
            Config.ARCH,
            pretrained=False,
            num_classes=0
        )

        ck = torch.load(
            IJEPA_CKPT,
            map_location="cpu"
        )

        enc.load_state_dict(
            ck["encoder_state_dict"],
            strict=True
        )

    else:
        raise ValueError(kind)

    # B2 DIFFERENCE:
    # encoder is trainable rather than frozen
    for p in enc.parameters():
        p.requires_grad = True

    return enc.to(DEVICE)

class ViTSegModel(nn.Module):
    def __init__(self,enc):
        super().__init__()
        self.encoder=enc
        d=enc.num_features
        self.decoder=nn.Sequential(
            nn.Conv2d(d,256,3,padding=1), nn.ReLU(),
            nn.Conv2d(256,128,3,padding=1), nn.ReLU(),
            nn.Conv2d(128,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,1,1)
        )
    def forward(self,x):
        with torch.no_grad():
            z=self.encoder.forward_features(x)
        z=z[:,getattr(self.encoder,"num_prefix_tokens",1):,:]
        B,N,C=z.shape
        s=int(N**0.5)
        assert s*s==N
        z=z.transpose(1,2).reshape(B,C,s,s)
        return F.interpolate(self.decoder(z),size=(Config.IMG_SIZE,Config.IMG_SIZE),
                             mode="bilinear",align_corners=False)

# EXPERIMENT B2 — FINE-TUNING SEGMENTATION MODEL
# Same decoder as frozen B, but gradients flow through encoder

class ViTSegModelFineTune(nn.Module):
    def __init__(self, enc):
        super().__init__()

        self.encoder = enc
        d = enc.num_features

        # EXACT SAME DECODER AS FROZEN EXPERIMENT B
        self.decoder = nn.Sequential(
            nn.Conv2d(d, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):

        # no torch.no_grad() here because encoder is fine-tuned
        z = self.encoder.forward_features(x)

        z = z[
            :,
            getattr(
                self.encoder,
                "num_prefix_tokens",
                1
            ):,
            :
        ]

        B, N, C = z.shape
        s = int(N ** 0.5)

        assert s * s == N

        z = (
            z.transpose(1, 2)
             .reshape(B, C, s, s)
        )

        return F.interpolate(
            self.decoder(z),
            size=(
                Config.IMG_SIZE,
                Config.IMG_SIZE
            ),
            mode="bilinear",
            align_corners=False
        )

def dice_loss(logits,t,eps=1e-6):
    p=torch.sigmoid(logits)
    inter=(p*t).sum((1,2,3))
    den=p.sum((1,2,3))+t.sum((1,2,3))
    return 1-((2*inter+eps)/(den+eps)).mean()

def loss_fn(logits,t):
    pw=torch.tensor([Config.POS_WEIGHT],device=logits.device)
    return F.binary_cross_entropy_with_logits(logits,t,pos_weight=pw)+dice_loss(logits,t)

print("Encoder conditions:", ENCODERS)


Encoder conditions: ['Random', 'ImageNet', 'MAE_clean_2840', 'IJEPA_clean_2840']


In [ ]:
# Experiment B2 — trainable encoder check

for kind in FT_ENCODERS:

    enc = build_finetune_encoder(
        kind,
        seed=42
    )

    n_total = sum(
        p.numel()
        for p in enc.parameters()
    )

    n_trainable = sum(
        p.numel()
        for p in enc.parameters()
        if p.requires_grad
    )

    print(
        f"{kind}: "
        f"total={n_total:,} | "
        f"trainable={n_trainable:,}"
    )

    assert n_trainable == n_total

    del enc

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nB2 trainable encoder audit PASSED.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

ImageNet: total=21,665,664 | trainable=21,665,664
IJEPA_clean_2840: total=21,665,664 | trainable=21,665,664

B2 trainable encoder audit PASSED.


## 5. Validation and test metrics

In [ ]:
@torch.no_grad()
def positive_dice(model,loader):
    model.eval()
    vals=[]
    for x,t in loader:
        p=(torch.sigmoid(model(x.to(DEVICE,non_blocking=True)))>=.5).float()
        t=t.to(DEVICE,non_blocking=True)
        for a,b in zip(p,t):
            if b.sum()==0:
                continue
            den=a.sum().item()+b.sum().item()
            vals.append(2*(a*b).sum().item()/den if den else 0.0)
    return float(np.mean(vals)) if vals else 0.0

@torch.no_grad()
def detailed(model,loader):
    model.eval()
    D,I,P,R=[],[],[],[]
    pred_pos=0
    total=0
    for x,t in loader:
        pred=(torch.sigmoid(model(x.to(DEVICE,non_blocking=True)))>=.5).float()
        t=t.to(DEVICE,non_blocking=True)
        for a,b in zip(pred,t):
            total+=1
            pred_pos+=int(a.sum().item()>0)
            if b.sum()==0:
                continue
            tp=(a*b).sum().item()
            fp=(a*(1-b)).sum().item()
            fn=((1-a)*b).sum().item()
            D.append(2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0)
            I.append(tp/(tp+fp+fn) if (tp+fp+fn) else 0)
            P.append(tp/(tp+fp) if (tp+fp) else 0)
            R.append(tp/(tp+fn) if (tp+fn) else 0)
    return {
        "test_positive_dice":float(np.mean(D)),
        "test_positive_iou":float(np.mean(I)),
        "test_positive_precision":float(np.mean(P)),
        "test_positive_recall":float(np.mean(R)),
        "pred_positive_images":pred_pos,
        "total_test_images":total
    }


## 6. Resume-safe single-run training

Each run saves its latest epoch state to Drive. If Colab disconnects, rerunning the same encoder/seed resumes from the next epoch. Completed runs are appended to the result file.

In [ ]:
def run_id(kind,seed):
    return f"B_{kind}_seed{seed}_frozen"

def state_path(kind,seed):
    safe=run_id(kind,seed).replace("/","_")
    return os.path.join(Config.STATE_DIR, safe+"_latest.pth")

def load_results():
    return pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame()

def completed(df,kind,seed):
    if len(df)==0:
        return False
    return bool(((df["encoder"]==kind)&(df["seed"]==seed)).any())

def save_result(r):
    df=load_results()
    if len(df):
        keep=~((df["encoder"]==r["encoder"])&(df["seed"]==r["seed"]))
        df=df.loc[keep].copy()
    df=pd.concat([df,pd.DataFrame([r])],ignore_index=True)
    df.to_csv(RESULT_CSV,index=False)
    return df

def run_B(kind, seed):
    """
    Frozen-encoder Experiment B run.

    Saves:
      1. Resume state after every epoch
      2. Per-epoch training loss
      3. Per-epoch positive-case validation Dice
      4. Best validation-selected model state

    Encoder remains frozen throughout.
    """

    set_seed(seed)

    # DataLoaders are reconstructed for every encoder/seed run
    train_loader, val_loader, test_loader = make_loaders(seed)

    # Build frozen encoder + segmentation model
    encoder = build_encoder(kind, seed)

    for p in encoder.parameters():
        p.requires_grad = False

    encoder.eval()

    #model = FrozenSegModel(encoder).to(DEVICE)
    model = ViTSegModel(encoder).to(DEVICE)

    # Only train parameters that require gradients
    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    opt = torch.optim.AdamW(
        trainable_params,
        lr=Config.LR,
        weight_decay=Config.WEIGHT_DECAY
    )

    # Resume/history paths
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    state_path = os.path.join(
        B_DEFINITIVE_STATE_DIR,
        f"B_{safe_kind}_seed{seed}_frozen_latest.pth"
    )

    history_csv = os.path.join(
        B_DEFINITIVE_HISTORY_DIR,
        f"experiment_B_history_{safe_kind}_seed{seed}.csv"
    )
    # Initial state
    start_epoch = 1
    best_val = -1.0
    best_epoch = None
    best_state = None

    history = []

    epochs_without_improvement = 0

    # Resume if interrupted
    if os.path.exists(state_path):

        print(f"Resume state found: {state_path}")

        state = torch.load(
            state_path,
            map_location=DEVICE
        )

        model.load_state_dict(
            state["model_state_dict"]
        )

        opt.load_state_dict(
            state["optimizer_state_dict"]
        )

        start_epoch = int(state["epoch"]) + 1

        best_val = float(
            state.get(
                "best_val_positive_dice",
                -1.0
            )
        )

        best_epoch = state.get(
            "best_epoch",
            None
        )

        best_state = state.get(
            "best_model_state_dict",
            None
        )

        # NEW: restore epoch history
        history = state.get(
            "history",
            []
        )
        epochs_without_improvement = int(
            state.get("epochs_without_improvement", 0)
        )
        print(
            f"Resuming {kind}, seed={seed} "
            f"from epoch {start_epoch}"
        )

        print(
            f"Current best validation Dice="
            f"{best_val:.4f} "
            f"(epoch {best_epoch})"
        )

    # Training
    for epoch in range(
        start_epoch,
        Config.MAX_EPOCHS + 1
    ):

        model.train()

        # Keep frozen encoder in evaluation mode
        model.encoder.eval()

        running_loss = 0.0
        n_batches = 0

        for images, masks in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            loss = loss_fn(
                logits,
                masks
            )

            loss.backward()
            opt.step()

            running_loss += loss.item()
            n_batches += 1

        train_loss = (
            running_loss /
            max(n_batches, 1)
        )

        # Validation: same positive-case Dice criterion
        val_dice = positive_dice(
            model,
            val_loader
        )

        # NEW: record every epoch
        epoch_record = {
            "encoder": kind,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_positive_dice": val_dice
        }

        history.append(epoch_record)

        # Best validation checkpoint

        if val_dice > best_val:

            best_val = val_dice
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        # Save resume state after EVERY epoch
        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    opt.state_dict(),

                "best_val_positive_dice":
                    best_val,

                "best_epoch":
                    best_epoch,

                "best_model_state_dict":
                    best_state,

                "encoder":
                    kind,

                "seed":
                    seed,

                # NEW
                "history":
                    history,
                "epochs_without_improvement":
                    epochs_without_improvement
            },
            state_path
        )

        # NEW: save readable history CSV after every epoch
        pd.DataFrame(
            history
        ).to_csv(
            history_csv,
            index=False
        )

        print(
            f"{kind:18s} | "
            f"seed {seed} | "
            f"epoch {epoch:02d}/{Config.MAX_EPOCHS} | "
            f"train loss {train_loss:.4f} | "
            f"val +Dice {val_dice:.4f} | "
            f"best {best_val:.4f} "
            f"@ epoch {best_epoch}"
        )
        if epochs_without_improvement >= Config.PATIENCE:

            print(
                f"EARLY STOPPING | "
                f"{kind} | seed {seed} | "
                f"epoch {epoch} | "
                f"best val +Dice {best_val:.4f} "
                f"@ epoch {best_epoch}"
            )

            break
    # Safety check
    assert best_state is not None, \
        f"No best model state found for {kind}, seed={seed}"

    # Reload validation-selected checkpoint
    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    # NEW: permanently save validation-selected checkpoint
    best_checkpoint_path = save_B_best_checkpoint(
        model=model,
        kind=kind,
        seed=seed,
        best_epoch=best_epoch,
        best_val=best_val
    )

    print(
        f"\nPermanent best checkpoint saved:\n"
        f"{best_checkpoint_path}"
    )

    # ONE final positive-case test evaluation
    # Same evaluation as original Experiment B
    test_metrics = detailed(
        model,
        test_loader
    )

    # NEW: negative-case false-positive evaluation
    # Same fixed test set; no model fitting or selection
    negative_metrics = negative_case_metrics(
        model,
        test_loader,
        threshold=0.5
    )

    print(
        f"\nFINAL TEST | "
        f"{kind} | seed {seed} | "
        f"best epoch={best_epoch} | "
        f"best val +Dice={best_val:.4f}"
    )

    print("\nPositive-case metrics:")
    print(test_metrics)

    print("\nNegative-case metrics:")
    print(negative_metrics)

    # Return result for master results CSV
    result = {
        "run_id":
            f"B_{safe_kind}_seed{seed}_frozen",

        "encoder":
            kind,

        "seed":
            seed,

        "frozen_encoder":
            True,

        "n_train":
            len(train_loader.dataset),

        "n_val":
            len(val_loader.dataset),

        "n_test":
            len(test_loader.dataset),

        "positive_test_cases":
            24,

        "negative_test_cases":
            24,

        "segmentation_threshold":
            0.5,

        "pos_weight":
            Config.POS_WEIGHT,

        "max_epochs":
            Config.MAX_EPOCHS,

        "patience":
            Config.PATIENCE,

        "best_epoch":
            best_epoch,

        "best_val_positive_dice":
            best_val,

        "best_checkpoint":
            best_checkpoint_path,

        # Original positive-case metrics
        **test_metrics,

        # New negative-case metrics
        **negative_metrics
    }

    return result

print("Resume-safe training function ready.")


Resume-safe training function ready.


In [ ]:
# Experiment B2 — best-checkpoint saver

def save_B_finetune_best_checkpoint(
    model,
    kind,
    seed,
    best_epoch,
    best_val
):
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    path = os.path.join(
        FT_CHECKPOINT_DIR,
        f"B2_{safe_kind}_seed{seed}_finetuned_best.pth"
    )

    torch.save(
        {
            "experiment": "B2_finetuning",
            "encoder": kind,
            "seed": seed,
            "frozen_encoder": False,
            "encoder_lr": FT_ENCODER_LR,
            "decoder_lr": FT_DECODER_LR,
            "best_epoch": int(best_epoch),
            "best_val_positive_dice": float(best_val),
            "arch": Config.ARCH,
            "model_state_dict": {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        },
        path
    )

    return path


print("B2 permanent checkpoint saver ready.")

def run_B_finetune(kind, seed):
    """
    Fine-tuned Experiment B2 run.

    ImageNet or clean I-JEPA encoder is trainable.
    Decoder architecture and downstream evaluation protocol
    are preserved from frozen Experiment B.
    """

    set_seed(seed)

    # Same DataLoaders as frozen Experiment B
    train_loader, val_loader, test_loader = make_loaders(seed)

    # Build trainable encoder + same segmentation decoder
    encoder = build_finetune_encoder(kind, seed)

    model = ViTSegModelFineTune(
        encoder
    ).to(DEVICE)

    # Differential learning rates
    opt = torch.optim.AdamW(
        [
            {
                "params": model.encoder.parameters(),
                "lr": FT_ENCODER_LR
            },
            {
                "params": model.decoder.parameters(),
                "lr": FT_DECODER_LR
            }
        ],
        weight_decay=FT_WEIGHT_DECAY
    )

    print(
        f"{kind} | seed={seed} | "
        f"encoder LR={FT_ENCODER_LR} | "
        f"decoder LR={FT_DECODER_LR}"
    )

    # B2 resume/history paths
    safe_kind = kind.replace("/", "_").replace(" ", "_")

    state_path = os.path.join(
        FT_STATE_DIR,
        f"B2_{safe_kind}_seed{seed}_finetuned_latest.pth"
    )

    history_csv = os.path.join(
        FT_HISTORY_DIR,
        f"B2_history_{safe_kind}_seed{seed}.csv"
    )

    # Initial state
    start_epoch = 1
    best_val = -1.0
    best_epoch = None
    best_state = None
    history = []
    epochs_without_improvement = 0

    # Resume if interrupted
    if os.path.exists(state_path):

        print(f"Resume state found: {state_path}")

        state = torch.load(
            state_path,
            map_location=DEVICE
        )

        model.load_state_dict(
            state["model_state_dict"]
        )

        opt.load_state_dict(
            state["optimizer_state_dict"]
        )

        start_epoch = int(state["epoch"]) + 1

        best_val = float(
            state.get(
                "best_val_positive_dice",
                -1.0
            )
        )

        best_epoch = state.get(
            "best_epoch",
            None
        )

        best_state = state.get(
            "best_model_state_dict",
            None
        )

        history = state.get(
            "history",
            []
        )

        epochs_without_improvement = int(
            state.get(
                "epochs_without_improvement",
                0
            )
        )

        print(
            f"Resuming {kind}, seed={seed} "
            f"from epoch {start_epoch}"
        )

        print(
            f"Current best validation Dice="
            f"{best_val:.4f} "
            f"(epoch {best_epoch})"
        )

    # Fine-tuning
    for epoch in range(
        start_epoch,
        FT_MAX_EPOCHS + 1
    ):

        # BOTH encoder and decoder are in training mode
        model.train()

        running_loss = 0.0
        n_batches = 0

        for images, masks in train_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            opt.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            # EXACT SAME LOSS FUNCTION AS FROZEN B
            loss = loss_fn(
                logits,
                masks
            )

            loss.backward()
            opt.step()

            running_loss += loss.item()
            n_batches += 1

        train_loss = (
            running_loss /
            max(n_batches, 1)
        )

        # SAME validation criterion as frozen B
        # Positive-mask validation Dice only
        val_dice = positive_dice(
            model,
            val_loader
        )

        epoch_record = {
            "encoder": kind,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_positive_dice": val_dice,
            "encoder_lr": FT_ENCODER_LR,
            "decoder_lr": FT_DECODER_LR
        }

        history.append(epoch_record)

        # Validation-selected best checkpoint
        if val_dice > best_val:

            best_val = val_dice
            best_epoch = epoch

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        # Resume state after every epoch
        torch.save(
            {
                "experiment":
                    "B2_finetuning",

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    opt.state_dict(),

                "best_val_positive_dice":
                    best_val,

                "best_epoch":
                    best_epoch,

                "best_model_state_dict":
                    best_state,

                "encoder":
                    kind,

                "seed":
                    seed,

                "frozen_encoder":
                    False,

                "encoder_lr":
                    FT_ENCODER_LR,

                "decoder_lr":
                    FT_DECODER_LR,

                "history":
                    history,

                "epochs_without_improvement":
                    epochs_without_improvement
            },
            state_path
        )

        # Human-readable history after every epoch
        pd.DataFrame(
            history
        ).to_csv(
            history_csv,
            index=False
        )

        print(
            f"{kind:18s} | "
            f"seed {seed} | "
            f"epoch {epoch:02d}/{FT_MAX_EPOCHS} | "
            f"train loss {train_loss:.4f} | "
            f"val +Dice {val_dice:.4f} | "
            f"best {best_val:.4f} "
            f"@ epoch {best_epoch}"
        )

        if epochs_without_improvement >= FT_PATIENCE:

            print(
                f"EARLY STOPPING | "
                f"{kind} | seed {seed} | "
                f"epoch {epoch} | "
                f"best val +Dice {best_val:.4f} "
                f"@ epoch {best_epoch}"
            )

            break

    # Safety check
    assert best_state is not None, \
        f"No best model state found for {kind}, seed={seed}"

    # Restore validation-selected model
    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    # Permanently preserve complete fine-tuned model
    best_checkpoint_path = (
        save_B_finetune_best_checkpoint(
            model=model,
            kind=kind,
            seed=seed,
            best_epoch=best_epoch,
            best_val=best_val
        )
    )

    print(
        f"\nPermanent B2 checkpoint saved:\n"
        f"{best_checkpoint_path}"
    )

    # SAME positive-case test evaluation as frozen B
    test_metrics = detailed(
        model,
        test_loader
    )

    # SAME negative-case evaluation as frozen B
    # Fixed threshold = 0.5
    negative_metrics = negative_case_metrics(
        model,
        test_loader,
        threshold=FT_THRESHOLD
    )

    print(
        f"\nFINAL B2 TEST | "
        f"{kind} | seed {seed} | "
        f"best epoch={best_epoch} | "
        f"best val +Dice={best_val:.4f}"
    )

    print("\nPositive-case metrics:")
    print(test_metrics)

    print("\nNegative-case metrics:")
    print(negative_metrics)

    # result row
    result = {
        "run_id":
            f"B2_{safe_kind}_seed{seed}_finetuned",

        "experiment":
            "B2_finetuning",

        "encoder":
            kind,

        "seed":
            seed,

        "frozen_encoder":
            False,

        "encoder_lr":
            FT_ENCODER_LR,

        "decoder_lr":
            FT_DECODER_LR,

        "weight_decay":
            FT_WEIGHT_DECAY,

        "n_train":
            len(train_loader.dataset),

        "n_val":
            len(val_loader.dataset),

        "n_test":
            len(test_loader.dataset),

        "positive_test_cases":
            24,

        "negative_test_cases":
            24,

        "segmentation_threshold":
            FT_THRESHOLD,

        "pos_weight":
            FT_POS_WEIGHT,

        "max_epochs":
            FT_MAX_EPOCHS,

        "patience":
            FT_PATIENCE,

        "best_epoch":
            best_epoch,

        "best_val_positive_dice":
            best_val,

        "best_checkpoint":
            best_checkpoint_path,

        **test_metrics,
        **negative_metrics
    }

    return result


print("Experiment B2 fine-tuning function ready.")

B2 permanent checkpoint saver ready.
Experiment B2 fine-tuning function ready.


In [ ]:
# Experiment B2 — result handling

def load_B2_results():
    if os.path.exists(FT_RESULT_CSV):
        return pd.read_csv(FT_RESULT_CSV)

    return pd.DataFrame()


def save_B2_result(result):
    df = load_B2_results()

    run_id = result["run_id"]

    # Prevent duplicate rows if a completed run is encountered again
    if len(df) and "run_id" in df.columns:
        df = df[
            df["run_id"] != run_id
        ].copy()

    df = pd.concat(
        [
            df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    df.to_csv(
        FT_RESULT_CSV,
        index=False
    )

    return df


print("B2 result handling ready.")
print("Result CSV:")
print(FT_RESULT_CSV)

B2 result handling ready.
Result CSV:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/results/experiment_B_finetuning_results.csv


In [ ]:
# Experiment B2 — setup check

import os
import glob
import torch

print("=" * 70)
print("EXPERIMENT B2 — FINE-TUNING PRE-LAUNCH AUDIT")
print("=" * 70)

# 1. Experimental design
assert FT_ENCODERS == [
    "ImageNet",
    "IJEPA_clean_2840"
]

assert FT_SEEDS == [42, 43, 44]

assert len(expected_ft_runs) == 6

print("\n[1] Design")
print("Encoders:", FT_ENCODERS)
print("Seeds:", FT_SEEDS)
print("Expected runs:", len(expected_ft_runs))


# 2. Dataset split
n_train = len(train_pairs_all) - len(val_pairs)
n_val = len(val_pairs)
n_test = len(test_pairs)

assert n_train == 394
assert n_val == 99
assert n_test == 48

print("\n[2] Ar-PlaqSegm1 split")
print(
    "Train =", n_train,
    "| Val =", n_val,
    "| Test =", n_test
)


# 3. Starting checkpoints / architecture
assert os.path.exists(IJEPA_CKPT)
assert Config.ARCH == "vit_small_patch16_224"

print("\n[3] Initialization")
print("Architecture:", Config.ARCH)
print("ImageNet: timm pretrained weights")
print("I-JEPA:", IJEPA_CKPT)


# 4. Fine-tuning hyperparameters
assert FT_ENCODER_LR == 1e-5
assert FT_DECODER_LR == 1e-3
assert FT_WEIGHT_DECAY == 1e-4
assert FT_MAX_EPOCHS == 50
assert FT_PATIENCE == 10
assert FT_POS_WEIGHT == 10.0
assert FT_THRESHOLD == 0.5

print("\n[4] Fine-tuning protocol")
print("Encoder LR:", FT_ENCODER_LR)
print("Decoder LR:", FT_DECODER_LR)
print("Weight decay:", FT_WEIGHT_DECAY)
print("Max epochs:", FT_MAX_EPOCHS)
print("Patience:", FT_PATIENCE)
print("Positive weight:", FT_POS_WEIGHT)
print("Threshold:", FT_THRESHOLD)


# 5. B2 namespace status
existing_results = load_B2_results()

checkpoint_files = glob.glob(
    os.path.join(
        FT_CHECKPOINT_DIR,
        "*.pth"
    )
)

resume_files = glob.glob(
    os.path.join(
        FT_STATE_DIR,
        "*.pth"
    )
)

history_files = glob.glob(
    os.path.join(
        FT_HISTORY_DIR,
        "*.csv"
    )
)

print("\n[5] Existing B2 artifacts")
print("Completed result rows:", len(existing_results), "/ 6")
print("Permanent checkpoints:", len(checkpoint_files))
print("Resume states:", len(resume_files))
print("History files:", len(history_files))


# 6. Gradient-flow test
print("\n[6] Encoder gradient-flow test")

test_encoder = build_finetune_encoder(
    "IJEPA_clean_2840",
    seed=42
)

test_model = ViTSegModelFineTune(
    test_encoder
).to(DEVICE)

test_model.train()

x_test = torch.randn(
    1,
    3,
    Config.IMG_SIZE,
    Config.IMG_SIZE,
    device=DEVICE
)

out_test = test_model(x_test)

# Simple artificial scalar objective solely to test autograd
out_test.mean().backward()

encoder_gradients = [
    p.grad
    for p in test_model.encoder.parameters()
    if p.requires_grad
]

n_with_grad = sum(
    g is not None
    for g in encoder_gradients
)

assert n_with_grad > 0, \
    "No gradients reached the encoder."

print(
    "Encoder parameter tensors receiving gradients:",
    n_with_grad
)

print("Gradient flow: PASSED")

del test_model
del test_encoder
del x_test
del out_test

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# verdict
print("\n" + "=" * 70)
print("B2 PRE-LAUNCH AUDIT PASSED")
print("=" * 70)

print(
    "Ready for 6 fine-tuning runs: "
    "ImageNet vs I-JEPA × seeds 42, 43, 44."
)

EXPERIMENT B2 — FINE-TUNING PRE-LAUNCH AUDIT

[1] Design
Encoders: ['ImageNet', 'IJEPA_clean_2840']
Seeds: [42, 43, 44]
Expected runs: 6

[2] Ar-PlaqSegm1 split
Train = 394 | Val = 99 | Test = 48

[3] Initialization
Architecture: vit_small_patch16_224
ImageNet: timm pretrained weights
I-JEPA: /content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/checkpoints/ijepa_corrected_2840_encoder.pth

[4] Fine-tuning protocol
Encoder LR: 1e-05
Decoder LR: 0.001
Weight decay: 0.0001
Max epochs: 50
Patience: 10
Positive weight: 10.0
Threshold: 0.5

[5] Existing B2 artifacts
Completed result rows: 0 / 6
Permanent checkpoints: 0
Resume states: 0
History files: 0

[6] Encoder gradient-flow test
Encoder parameter tensors receiving gradients: 150
Gradient flow: PASSED

B2 PRE-LAUNCH AUDIT PASSED
Ready for 6 fine-tuning runs: ImageNet vs I-JEPA × seeds 42, 43, 44.


In [ ]:
# Experiment B2 — 6-run fine-tuning launcher
# ImageNet vs EMA-distillation × seeds 42, 43, 44 (historical artifact name retained in variables)

print("=" * 70)
print("EXPERIMENT B2 — FINE-TUNING")
print("=" * 70)

print("Expected runs:", len(expected_ft_runs))
print("Encoders:", FT_ENCODERS)
print("Seeds:", FT_SEEDS)
print()

for seed in FT_SEEDS:

    for kind in FT_ENCODERS:

        safe_kind = kind.replace("/", "_").replace(" ", "_")

        run_id = (
            f"B2_{safe_kind}_seed{seed}_finetuned"
        )

        # Skip only genuinely completed runs
        existing = load_B2_results()

        if (
            len(existing)
            and "run_id" in existing.columns
            and run_id in existing["run_id"].values
        ):
            print(
                f"SKIP completed B2 run: "
                f"{kind} | seed {seed}"
            )
            continue

        print()
        print("=" * 70)
        print(
            f"START B2 FINE-TUNING | "
            f"{kind} | seed {seed}"
        )
        print("=" * 70)

        result = run_B_finetune(
            kind=kind,
            seed=seed
        )

        df_B2 = save_B2_result(
            result
        )

        print(
            f"\nCOMPLETED B2 RUN | "
            f"{kind} | seed {seed}"
        )

        print(
            f"Completed: "
            f"{len(df_B2)}/6"
        )

        # Free GPU memory before next initialization
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print()
print("=" * 70)
print("EXPERIMENT B2 LAUNCHER FINISHED")
print("=" * 70)

final_B2 = load_B2_results()

print(
    f"\nCompleted B2 runs: "
    f"{len(final_B2)}/6"
)

if len(final_B2):
    display(
        final_B2.sort_values(
            ["seed", "encoder"]
        )
    )

EXPERIMENT B2 — FINE-TUNING
Expected runs: 6
Encoders: ['ImageNet', 'IJEPA_clean_2840']
Seeds: [42, 43, 44]


START B2 FINE-TUNING | ImageNet | seed 42
ImageNet | seed=42 | encoder LR=1e-05 | decoder LR=0.001
ImageNet           | seed 42 | epoch 01/50 | train loss 1.3552 | val +Dice 0.0000 | best 0.0000 @ epoch 1
ImageNet           | seed 42 | epoch 02/50 | train loss 1.1427 | val +Dice 0.0000 | best 0.0000 @ epoch 1
ImageNet           | seed 42 | epoch 03/50 | train loss 1.0882 | val +Dice 0.2868 | best 0.2868 @ epoch 3
ImageNet           | seed 42 | epoch 04/50 | train loss 1.0097 | val +Dice 0.2908 | best 0.2908 @ epoch 4
ImageNet           | seed 42 | epoch 05/50 | train loss 0.9805 | val +Dice 0.3188 | best 0.3188 @ epoch 5
ImageNet           | seed 42 | epoch 06/50 | train loss 0.9090 | val +Dice 0.3306 | best 0.3306 @ epoch 6
ImageNet           | seed 42 | epoch 07/50 | train loss 0.8723 | val +Dice 0.2819 | best 0.3306 @ epoch 6
ImageNet           | seed 42 | epoch 08/50 | trai

,run_id,experiment,encoder,seed,frozen_encoder,encoder_lr,decoder_lr,weight_decay,n_train,n_val,...,test_positive_precision,test_positive_recall,pred_positive_images,total_test_images,n_negative_test,negative_cases_with_any_fp,false_positive_image_rate,mean_false_positive_pixel_fraction,sd_false_positive_pixel_fraction,median_false_positive_pixel_fraction
1,B2_IJEPA_clean_2840_seed42_finetuned,B2_finetuning,IJEPA_clean_2840,42,False,0.00001,0.001,0.0001,394,99,...,0.363624,0.775393,41,48,24,17,0.708333,0.009598,0.013931,0.002511
0,B2_ImageNet_seed42_finetuned,B2_finetuning,ImageNet,42,False,0.00001,0.001,0.0001,394,99,...,0.341461,0.657100,41,48,24,18,0.750000,0.008347,0.008691,0.007005
3,B2_IJEPA_clean_2840_seed43_finetuned,B2_finetuning,IJEPA_clean_2840,43,False,0.00001,0.001,0.0001,394,99,...,0.418904,0.719096,38,48,24,14,0.583333,0.004209,0.007786,0.000189
2,B2_ImageNet_seed43_finetuned,B2_finetuning,ImageNet,43,False,0.00001,0.001,0.0001,394,99,...,0.265488,0.538023,23,48,24,5,0.208333,0.003228,0.008348,0.000000
5,B2_IJEPA_clean_2840_seed44_finetuned,B2_finetuning,IJEPA_clean_2840,44,False,0.00001,0.001,0.0001,394,99,...,0.426540,0.666239,38,48,24,14,0.583333,0.004903,0.008774,0.000428
4,B2_ImageNet_seed44_finetuned,B2_finetuning,ImageNet,44,False,0.00001,0.001,0.0001,394,99,...,0.305639,0.628883,39,48,24,16,0.666667,0.007803,0.010776,0.003059


In [ ]:
import os
import pandas as pd

print("B2 results CSV:")
print(FT_RESULT_CSV)

print("\nExists:", os.path.exists(FT_RESULT_CSV))

if os.path.exists(FT_RESULT_CSV):
    df_B2 = pd.read_csv(FT_RESULT_CSV)

    print("Rows:", len(df_B2))
    print("Columns:", len(df_B2.columns))
    print("Size:", os.path.getsize(FT_RESULT_CSV), "bytes")

    display(
        df_B2.sort_values(
            ["seed", "encoder"]
        )
    )

B2 results CSV:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/results/experiment_B_finetuning_results.csv

Exists: True
Rows: 6
Columns: 32
Size: 3197 bytes


,run_id,experiment,encoder,seed,frozen_encoder,encoder_lr,decoder_lr,weight_decay,n_train,n_val,...,test_positive_precision,test_positive_recall,pred_positive_images,total_test_images,n_negative_test,negative_cases_with_any_fp,false_positive_image_rate,mean_false_positive_pixel_fraction,sd_false_positive_pixel_fraction,median_false_positive_pixel_fraction
1,B2_IJEPA_clean_2840_seed42_finetuned,B2_finetuning,IJEPA_clean_2840,42,False,0.00001,0.001,0.0001,394,99,...,0.363624,0.775393,41,48,24,17,0.708333,0.009598,0.013931,0.002511
0,B2_ImageNet_seed42_finetuned,B2_finetuning,ImageNet,42,False,0.00001,0.001,0.0001,394,99,...,0.341461,0.657100,41,48,24,18,0.750000,0.008347,0.008691,0.007005
3,B2_IJEPA_clean_2840_seed43_finetuned,B2_finetuning,IJEPA_clean_2840,43,False,0.00001,0.001,0.0001,394,99,...,0.418904,0.719096,38,48,24,14,0.583333,0.004209,0.007786,0.000189
2,B2_ImageNet_seed43_finetuned,B2_finetuning,ImageNet,43,False,0.00001,0.001,0.0001,394,99,...,0.265488,0.538023,23,48,24,5,0.208333,0.003228,0.008348,0.000000
5,B2_IJEPA_clean_2840_seed44_finetuned,B2_finetuning,IJEPA_clean_2840,44,False,0.00001,0.001,0.0001,394,99,...,0.426540,0.666239,38,48,24,14,0.583333,0.004903,0.008774,0.000428
4,B2_ImageNet_seed44_finetuned,B2_finetuning,ImageNet,44,False,0.00001,0.001,0.0001,394,99,...,0.305639,0.628883,39,48,24,16,0.666667,0.007803,0.010776,0.003059


In [ ]:
# Experiment B2 — mean ± SD summary

import pandas as pd
import os

df = pd.read_csv(FT_RESULT_CSV)

assert len(df) == 6, \
    f"Expected 6 B2 runs, found {len(df)}"

metrics = [
    "test_positive_dice",
    "test_positive_iou",
    "test_positive_precision",
    "test_positive_recall",
    "false_positive_image_rate",
    "mean_false_positive_pixel_fraction",
    "median_false_positive_pixel_fraction"
]

summary_B2 = (
    df.groupby("encoder")[metrics]
      .agg(["mean", "std"])
)

summary_B2.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary_B2.columns
]

summary_B2 = summary_B2.reset_index()

encoder_order = [
    "ImageNet",
    "IJEPA_clean_2840"
]

summary_B2["encoder"] = pd.Categorical(
    summary_B2["encoder"],
    categories=encoder_order,
    ordered=True
)

summary_B2 = (
    summary_B2
    .sort_values("encoder")
    .reset_index(drop=True)
)

summary_B2.to_csv(
    FT_SUMMARY_CSV,
    index=False
)

print("Saved B2 mean ± SD summary:")
print(FT_SUMMARY_CSV)

print("\nRows:", len(summary_B2))
print("Expected: 2")

display(summary_B2)

Saved B2 mean ± SD summary:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/results/experiment_B_finetuning_summary_mean_sd.csv

Rows: 2
Expected: 2


,encoder,test_positive_dice_mean,test_positive_dice_std,test_positive_iou_mean,test_positive_iou_std,test_positive_precision_mean,test_positive_precision_std,test_positive_recall_mean,test_positive_recall_std,false_positive_image_rate_mean,false_positive_image_rate_std,mean_false_positive_pixel_fraction_mean,mean_false_positive_pixel_fraction_std,median_false_positive_pixel_fraction_mean,median_false_positive_pixel_fraction_std
0,ImageNet,0.367720,0.028479,0.248517,0.016274,0.304196,0.038007,0.608002,0.062224,0.541667,0.291667,0.006459,0.002812,0.003355,0.003512
1,IJEPA_clean_2840,0.467305,0.015812,0.317324,0.017157,0.403022,0.034333,0.720243,0.054586,0.625000,0.072169,0.006236,0.002932,0.001043,0.001277


In [ ]:
# Experiment B2 — checkpoint check

import glob
import torch
import pandas as pd

checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            FT_CHECKPOINT_DIR,
            "B2_*_finetuned_best.pth"
        )
    )
)

print("=" * 70)
print("EXPERIMENT B2 — CHECKPOINT AUDIT")
print("=" * 70)

print("\nCheckpoint directory:")
print(FT_CHECKPOINT_DIR)

print(
    f"\nPermanent checkpoints found: "
    f"{len(checkpoint_files)}/6"
)

audit_rows = []

for path in checkpoint_files:

    size_mb = os.path.getsize(path) / (1024 ** 2)

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        state = ckpt.get(
            "model_state_dict",
            {}
        )

        row = {
            "file": os.path.basename(path),
            "encoder": ckpt.get("encoder"),
            "seed": ckpt.get("seed"),
            "frozen_encoder":
                ckpt.get("frozen_encoder"),
            "encoder_lr":
                ckpt.get("encoder_lr"),
            "decoder_lr":
                ckpt.get("decoder_lr"),
            "best_epoch":
                ckpt.get("best_epoch"),
            "best_val_positive_dice":
                ckpt.get(
                    "best_val_positive_dice"
                ),
            "arch": ckpt.get("arch"),
            "size_MB": size_mb,
            "n_state_tensors": len(state),
            "readable": True
        }

    except Exception as e:

        row = {
            "file": os.path.basename(path),
            "readable": False,
            "error": str(e)
        }

    audit_rows.append(row)


audit_B2 = pd.DataFrame(audit_rows)

display(audit_B2)


# Expected encoder × seed combinations

expected_pairs = {
    (encoder, seed)
    for encoder in FT_ENCODERS
    for seed in FT_SEEDS
}

if len(audit_B2):

    actual_pairs = set(
        zip(
            audit_B2["encoder"],
            audit_B2["seed"]
        )
    )

else:
    actual_pairs = set()


missing = expected_pairs - actual_pairs
unexpected = actual_pairs - expected_pairs


# Additional fine-tuning provenance checks

all_readable = (
    len(audit_B2) == 6
    and audit_B2["readable"].all()
)

all_finetuned = (
    len(audit_B2) == 6
    and (
        audit_B2["frozen_encoder"]
        == False
    ).all()
)

correct_encoder_lr = (
    len(audit_B2) == 6
    and (
        audit_B2["encoder_lr"]
        == FT_ENCODER_LR
    ).all()
)

correct_decoder_lr = (
    len(audit_B2) == 6
    and (
        audit_B2["decoder_lr"]
        == FT_DECODER_LR
    ).all()
)


print("\n" + "=" * 70)

if (
    len(checkpoint_files) == 6
    and all_readable
    and all_finetuned
    and correct_encoder_lr
    and correct_decoder_lr
    and not missing
    and not unexpected
):

    print("B2 AUDIT PASSED")
    print(
        "All 6 permanent fine-tuned checkpoints "
        "are present and readable."
    )
    print(
        "Encoder LR and decoder LR provenance "
        "are correct."
    )

else:

    print("B2 AUDIT FAILED")

    if missing:
        print("\nMissing:")
        for item in sorted(missing):
            print(item)

    if unexpected:
        print("\nUnexpected:")
        for item in sorted(unexpected):
            print(item)

EXPERIMENT B2 — CHECKPOINT AUDIT

Checkpoint directory:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/checkpoints

Permanent checkpoints found: 6/6


,file,encoder,seed,frozen_encoder,encoder_lr,decoder_lr,best_epoch,best_val_positive_dice,arch,size_MB,n_state_tensors,readable
0,B2_IJEPA_clean_2840_seed42_finetuned_best.pth,IJEPA_clean_2840,42,False,0.00001,0.001,9,0.385694,vit_small_patch16_224,87.483348,158,True
1,B2_IJEPA_clean_2840_seed43_finetuned_best.pth,IJEPA_clean_2840,43,False,0.00001,0.001,13,0.407031,vit_small_patch16_224,87.483348,158,True
2,B2_IJEPA_clean_2840_seed44_finetuned_best.pth,IJEPA_clean_2840,44,False,0.00001,0.001,10,0.398769,vit_small_patch16_224,87.483348,158,True
3,B2_ImageNet_seed42_finetuned_best.pth,ImageNet,42,False,0.00001,0.001,9,0.347408,vit_small_patch16_224,87.482097,158,True
4,B2_ImageNet_seed43_finetuned_best.pth,ImageNet,43,False,0.00001,0.001,21,0.375271,vit_small_patch16_224,87.482097,158,True
5,B2_ImageNet_seed44_finetuned_best.pth,ImageNet,44,False,0.00001,0.001,14,0.363690,vit_small_patch16_224,87.482097,158,True



B2 AUDIT PASSED
All 6 permanent fine-tuned checkpoints are present and readable.
Encoder LR and decoder LR provenance are correct.


In [ ]:
audit_B2.to_csv(
    FT_AUDIT_CSV,
    index=False
)

print("Saved B2 checkpoint audit:")
print(FT_AUDIT_CSV)

Saved B2 checkpoint audit:
/content/drive/MyDrive/IJEPA_Carotid_ViTS16/clean_2840_20260930/experiment_B_finetuning/results/experiment_B_finetuning_checkpoint_audit.csv


## Results discussion

After supervised fine-tuning, the EMA-distillation initialization retains a mean advantage over ImageNet initialization in the reported downstream segmentation metrics. Refer to the result CSV and manuscript for the reported values.

## Interpretation guardrails

Experiment B2 is an **independent supervised fine-tuning transfer experiment**, not a clinical diagnostic validation. It compares ImageNet initialization with the carotid-domain EMA-distillation initialization. Historical `IJEPA` artifact names in the code are retained for provenance.